# Study 2: Why do agents choose high environmental investment?

- Notebook 04 describes how agents divide actions between development, prevention, and cleanup. I want to understand the income and reward conditions around those choices.
- Does development raise income enough to support later environmental investment? Do agents stop developing after balances grow, or while balances are still rising?
- How do development gains and pollution damage contribute to reward at different stages? Does this help explain the choice of high investment levels?
- The selected economic reward combines EDI gain with `(income - base income) / 10`.

## What does the reward measure?

- A successful development action earns its EDI gain as an immediate reward. Income above the base amount also adds a smaller reward each step, whether the region develops or not.
- Pollution reduces reward every step. Mitigation can improve reward by lowering pollution, even when the region is far from failure.
- Spending currency has no direct reward penalty; it limits which actions the region can afford later. Failed action attempts have a penalty.

## Settings

In [1]:
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.environment import (EconomicSustainabilityEnvironment, action_parts)
from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.train import load_q_tables

In [2]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))
environment_config = EnvironmentConfig()

## Load the saved policies

- One combined JSON file stores the four regional Q-tables for each reward mode and training seed.
- I will use the same greedy action choices as notebook 04 and record what happens around each choice.

In [3]:
agents_by_run = load_q_tables(REWARD_MODES, TRAINING_SEEDS, results_directory="../results", notebook_number=3)
print(f"Loaded {len(agents_by_run)} trained policies")

Loaded 15 trained policies


## Record what happens at each step

- One row will describe one region at one step. The four regions still act together.
- EDI and currency are recorded before the action. Income, cost, and reward come from that step.

In [4]:
def step_rows(step, actions, edi_before, currency_before, pollution_before, rewards, info, config):
    rows = []

    for region in range(len(config.regions)):
        development_choice, investment_choice = action_parts(actions[region])
        investment_type = "none"
        investment_level = 0

        if investment_choice in (1, 2, 3):
            investment_type = "prevention"
            investment_level = investment_choice
        elif investment_choice in (4, 5, 6):
            investment_type = "cleanup"
            investment_level = investment_choice - 3

        income = info["income"][region]
        income_reward = (income - config.income_base)
        income_reward /= config.ongoing_income_reward_scale
        economic_reward = config.economy_weight * (info["development_gains"][region] + income_reward)
        row = {
            "step": step,
            "region": region,
            "action": actions[region],
            "action_name": info["action_names"][region],
            "development": development_choice == 1,
            "investment_type": investment_type,
            "investment_level": investment_level,
            "valid_action": info["valid_actions"][region],
            "edi_before": edi_before[region],
            "edi_gain": info["development_gains"][region],
            "currency_before": currency_before[region],
            "pollution_before": pollution_before[region],
            "income": income,
            "development_cost": info["development_costs_paid"][region],
            "prevention_cost": info["prevention_costs_paid"][region],
            "cleanup_cost": info["cleanup_costs_paid"][region],
            "investment_cost": (info["prevention_costs_paid"][region] + info["cleanup_costs_paid"][region]),
            "cost_paid": info["costs_paid"][region],
            "economic_reward": economic_reward,
            "pollution_reward": (-config.pollution_weight * info["pollution_damage"][region]),
            "received_reward": rewards[region],
        }

        for pollutant in config.pollutants:
            gross_emissions = info["gross_emissions"][pollutant][region]
            remaining_emissions = info["remaining_emissions"][pollutant][region]
            row[f"gross_{pollutant}_emissions"] = gross_emissions
            row[f"remaining_{pollutant}_emissions"] = remaining_emissions
            row[f"avoided_{pollutant}_emissions"] = (gross_emissions - remaining_emissions)
            row[f"cleanup_{pollutant}_removed"] = (info["cleanup_removed"][pollutant][region])

        rows.append(row)

    return rows

In [5]:
def collect_episode(agents, reward_mode, seed):
    environment = EconomicSustainabilityEnvironment(reward_mode, environment_config)
    transitions, terminated, truncated = run_episode(environment, agents, seed)
    rows = []

    for transition in transitions:
        rows.extend(step_rows(
            transition["step"],
            transition["actions"],
            transition["development_before"],
            transition["currency_before"],
            transition["pollution_before"],
            transition["rewards"],
            transition["info"],
            environment.config,
        ))

    return rows

In [6]:
step_data = []

for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]

        for environment_seed in EVALUATION_SEEDS:
            rows = collect_episode(agents, reward_mode, environment_seed)
            for row in rows:
                row["reward_mode"] = reward_mode
                row["training_seed"] = training_seed
            step_data.extend(rows)

steps = pd.DataFrame(step_data)
print(f"Recorded {len(steps):,} region-steps")

Recorded 240,000 region-steps


## What changes over the episode?

- I summarize region-steps across the 20 held-out starts within each training seed, then average the five seed summaries.

In [7]:
steps["stage"] = 1 + (steps["step"] - 1) // 50
steps["development_executed"] = steps["development"] & steps["valid_action"]
steps["high_prevention"] = (steps["investment_type"].eq("prevention") & steps["investment_level"].eq(3) & steps["valid_action"])
steps["high_cleanup"] = (steps["investment_type"].eq("cleanup") & steps["investment_level"].eq(3) & steps["valid_action"])
steps["can_afford_high_prevention"] = steps["currency_before"].ge(environment_config.prevention_costs[3])
steps["currency_change"] = steps["income"] - steps["cost_paid"]

stage_by_seed = steps.groupby(["reward_mode", "training_seed", "stage"]).agg(
    develop_percent=("development_executed", "mean"),
    high_prevention_percent=("high_prevention", "mean"),
    high_cleanup_percent=("high_cleanup", "mean"),
    mean_edi=("edi_before", "mean"),
    mean_income=("income", "mean"),
    mean_currency=("currency_before", "mean"),
    currency_change_per_step=("currency_change", "mean"),
    can_afford_high_prevention_percent=("can_afford_high_prevention", "mean"),
    mean_economic_reward=("economic_reward", "mean"),
    mean_pollution_reward=("pollution_reward", "mean"),
    mean_received_reward=("received_reward", "mean"),
)

In [ ]:
for column in ("develop_percent", "high_prevention_percent", "high_cleanup_percent", "can_afford_high_prevention_percent"):
    stage_by_seed[column] *= 100

stage_summary = stage_by_seed.groupby(["reward_mode", "stage"]).mean()

### Actions and resources

- `mean_currency` is the balance before a decision.
- `currency_change_per_step` is income minus spending; a positive value means balances grow on average.
- `high_prevention_percent` and `high_cleanup_percent` show how often regions selected and successfully paid for the strongest prevention and cleanup levels.
- `can_afford_high_prevention_percent` measures whether the balance could cover high prevention by itself. It does not account for a simultaneous development cost.

In [8]:
resource_columns = [
    "develop_percent", "high_prevention_percent", "high_cleanup_percent",
    "mean_edi", "mean_income", "mean_currency",
    "currency_change_per_step", "can_afford_high_prevention_percent",
    ]

display(stage_summary[resource_columns].round(1))

develop_percent  high_prevention_percent  \
reward_mode stage                                             
global      1                 22.3                      5.0   
            2                  1.0                     15.8   
            3                  0.0                     15.2   
            4                  0.0                     10.0   
local       1                 27.6                     15.6   
            2                  2.2                     17.8   
            3                  0.0                     13.2   
            4                  0.0                     13.2   
mixed       1                 29.8                     12.0   
            2                  2.1                     18.4   
            3                  0.0                     14.6   
            4                  0.0                     14.6   

                   high_cleanup_percent  mean_edi  mean_income  mean_currency  \
reward_mode stage                                                               
global      1                       5.2      58.0         49.0          508.1   
            2                       7.4      73.0         56.5         1834.8   
            3                       7.4      73.2         56.6         3366.1   
            4                       8.1      73.2         56.6         4913.5   
local       1                       3.0      60.3         50.2          365.4   
            2                       7.7      78.3         59.1         1284.3   
            3                       9.7      79.0         59.5         2714.1   
            4                      10.2      79.0         59.5         4201.3   
mixed       1                       1.4      61.3         50.6          375.9   
            2                       6.6      81.5         60.8         1522.4   
            3                       7.6      81.9         61.0         3236.0   
            4                       7.6      81.9         61.0         4966.1   

                   currency_change_per_step  \
reward_mode stage                             
global      1                          10.7   
            2                          30.1   
            3                          30.6   
            4                          31.5   
local       1                           2.3   
            2                          25.9   
            3                          29.9   
            4                          29.6   
mixed       1                           4.3   
            2                          32.0   
            3                          34.7   
            4                          34.6   

                   can_afford_high_prevention_percent  
reward_mode stage                                      
global      1                                    99.6  
            2                                   100.0  
            3                                   100.0  
            4                                   100.0  
local       1                                    99.9  
            2                                   100.0  
            3                                   100.0  
            4                                   100.0  
mixed       1                                    99.9  
            2                                   100.0  
            3                                   100.0  
            4                                   100.0

- This table shows how development, prevention, cleanup, income, and currency change across each episode.
- I would compare the development share with the balance and currency change. If development falls while balances keep growing, the agents may already have enough income for their later choices.

### Reward components

- `mean_economic_reward` combines EDI gain with the ongoing-income bonus. `mean_pollution_reward` is the separate weighted pollution damage. These are local components before reward pooling.
- `mean_received_reward` is what each agent actually received under its reward mode. It can also include invalid-action and failure penalties.

In [9]:
reward_columns = ["mean_economic_reward", "mean_pollution_reward", "mean_received_reward"]
display(stage_summary[reward_columns].round(2))

mean_economic_reward  mean_pollution_reward  \
reward_mode stage                                                
global      1                      4.34                  -0.12   
            2                      3.67                  -0.15   
            3                      3.66                  -0.16   
            4                      3.66                  -0.17   
local       1                      4.55                  -0.13   
            2                      3.96                  -0.12   
            3                      3.95                  -0.13   
            4                      3.95                  -0.13   
mixed       1                      4.66                  -0.14   
            2                      4.12                  -0.17   
            3                      4.10                  -0.17   
            4                      4.10                  -0.17   

                   mean_received_reward  
reward_mode stage                        
global      1                      4.10  
            2                      3.46  
            3                      3.50  
            4                      3.49  
local       1                      4.36  
            2                      3.32  
            3                      2.90  
            4                      2.90  
mixed       1                      4.47  
            2                      3.75  
            3                      3.86  
            4                      3.86

- The economic component combines EDI gain with the ongoing-income reward. The pollution component is the negative damage from the four pollutant stocks.
- Spending does not have a direct reward term, although it changes which actions are affordable later. An environmental investment can affect reward through its effect on pollution damage.

## How do medium and high environmental investments compare?

- Prevention reduces new emissions. Cleanup removes some pollution that is already in the stocks.
- Levels 2 and 3 are the medium and high investment levels. The table compares their observed costs and effects, while keeping prevention and cleanup separate.

In [10]:
investment_comparison = steps[steps["valid_action"] & steps["investment_type"].isin(("prevention", "cleanup")) & steps["investment_level"].isin((2, 3))].copy()

investment_by_seed = investment_comparison.groupby(["reward_mode", "training_seed", "stage", "investment_type", "investment_level"]).agg(
    action_count=("action", "size"),
    mean_edi_before=("edi_before", "mean"),
    mean_currency_before=("currency_before", "mean"),
    mean_investment_cost=("investment_cost", "mean"),
    mean_development_cost=("development_cost", "mean"),
    mean_air_emissions_avoided=("avoided_air_emissions", "mean"),
    mean_water_emissions_avoided=("avoided_water_emissions", "mean"),
    mean_soil_emissions_avoided=("avoided_soil_emissions", "mean"),
    mean_noise_emissions_avoided=("avoided_noise_emissions", "mean"),
    mean_air_cleanup=("cleanup_air_removed", "mean"),
    mean_water_cleanup=("cleanup_water_removed", "mean"),
    mean_soil_cleanup=("cleanup_soil_removed", "mean"),
    mean_noise_cleanup=("cleanup_noise_removed", "mean"),
)

investment_summary = investment_by_seed.groupby(level=["reward_mode", "stage", "investment_type", "investment_level"]).mean()
display(investment_summary.round(2))

action_count  \
reward_mode stage investment_type investment_level                 
global      1     cleanup         2                       111.20   
                                  3                       208.20   
                  prevention      2                       415.20   
                                  3                       200.60   
            2     cleanup         2                       687.50   
                                  3                       368.25   
                  prevention      2                       949.33   
                                  3                       789.50   
            3     cleanup         2                       607.33   
                                  3                       490.00   
                  prevention      2                       950.67   
                                  3                      1014.67   
            4     cleanup         2                       737.67   
                                  3                       404.50   
                  prevention      2                       951.33   
                                  3                       667.33   
local       1     cleanup         2                       241.60   
                                  3                       119.80   
                  prevention      2                       510.00   
                                  3                       625.80   
            2     cleanup         2                       751.60   
                                  3                       516.00   
                  prevention      2                       895.60   
                                  3                       710.60   
            3     cleanup         2                      1333.33   
                                  3                       644.00   
                  prevention      2                       754.60   
                                  3                       880.67   
            4     cleanup         2                      1333.33   
                                  3                       679.00   
                  prevention      2                       748.40   
                                  3                       876.67   
mixed       1     cleanup         2                       235.60   
                                  3                        55.20   
                  prevention      2                       444.20   
                                  3                       479.20   
            2     cleanup         2                       773.00   
                                  3                       331.75   
                  prevention      2                       876.25   
                                  3                       734.80   
            3     cleanup         2                      1208.00   
                                  3                       380.25   
                  prevention      2                       947.75   
                                  3                       974.33   
            4     cleanup         2                      1231.00   
                                  3                       381.50   
                  prevention      2                       948.50   
                                  3                      1458.50   

                                                    mean_edi_before  \
reward_mode stage investment_type investment_level                    
global      1     cleanup         2                           32.79   
                                  3                           52.19   
                  prevention      2                           58.42   
                                  3                           54.56   
            2     cleanup         2                           70.00   
                                  3                           71.30   
                  prevention      2                           70.00   
            